# Phase 5: Multi-camera Re-ID crops (pseudo-labels)
GPU T4 on karo, multicam-videos dataset attach karo, fir Run All.

In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.7 MB/s eta 0:00:00


In [2]:
# Kaggle notebook ke ek cell me paste karo (GPU on). Pehle cell me: !pip install -q ultralytics
import os, re, glob, json, shutil
import cv2
import numpy as np
from ultralytics import YOLO

# ===== Settings =====
VIDEO_DIR = "/kaggle/input"                 # saare attached datasets me .mp4 dhoondega
OUT_DIR   = "/kaggle/working/multicam_crops"
SKIP      = ["chunk_032"]                   # TEST video kabhi training me nahi
EVERY     = 5      # har 5th frame ka crop (near-duplicate kam)
CONF      = 0.4    # sirf confident detections
MIN_H, MIN_W = 100, 40
MIN_IMG   = 5      # isse kam crops wali identity hatao
MAX_IMG   = 60     # ek identity ke max crops
MIN_MOVE  = 20     # pixels; isse kam hilne wala track = static false positive (poster etc.)
# ====================


def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0


videos = sorted(p for p in glob.glob(f"{VIDEO_DIR}/**/*.mp4", recursive=True)
                if not any(s in p for s in SKIP))
print(f"{len(videos)} videos mile:")
for v in videos:
    print("  ", v)

summary = {}
for vp in videos:
    stem = os.path.splitext(os.path.basename(vp))[0]
    m = re.search(r"Cam_\d+", stem, re.IGNORECASE)
    cam = m.group(0) if m else "unknown"
    folder = os.path.join(OUT_DIR, stem)
    counts, centers = {}, {}

    model = YOLO("yolov8n.pt")   # har video ke liye fresh tracker
    results = model.track(source=vp, stream=True, tracker="bytetrack.yaml",
                          classes=[0], conf=CONF, verbose=False)
    for fi, r in enumerate(results):
        if fi % EVERY or r.boxes.id is None:
            continue
        img = r.orig_img
        H, W = img.shape[:2]
        xyxy = r.boxes.xyxy.cpu().numpy()
        ids = r.boxes.id.int().cpu().numpy()
        for k, (b, tid) in enumerate(zip(xyxy, ids)):
            tid = int(tid)
            x1, y1, x2, y2 = b.astype(int)
            centers.setdefault(tid, []).append(((x1 + x2) / 2, (y1 + y2) / 2))
            if (y2 - y1) < MIN_H or (x2 - x1) < MIN_W:
                continue
            if x1 < 3 or y1 < 3 or x2 > W - 3 or y2 > H - 3:
                continue
            if any(iou(b, xyxy[j]) > 0.2 for j in range(len(xyxy)) if j != k):
                continue
            if counts.get(tid, 0) >= MAX_IMG:
                continue
            d = os.path.join(folder, str(tid))
            os.makedirs(d, exist_ok=True)
            cv2.imwrite(os.path.join(d, f"{fi}.jpg"), img[y1:y2, x1:x2])
            counts[tid] = counts.get(tid, 0) + 1
        if fi % 2000 == 0:
            print(f"  {stem}: frame {fi}, ids so far {len(counts)}")

    # kharab identities hatao: bahut kam crops, ya static (poster jaisa)
    kept, n_crops = 0, 0
    for tid, n in counts.items():
        c = np.array(centers[tid])
        moved = np.ptp(c[:, 0]) + np.ptp(c[:, 1]) if len(c) > 1 else 0
        d = os.path.join(folder, str(tid))
        if n < MIN_IMG or moved < MIN_MOVE:
            shutil.rmtree(d, ignore_errors=True)
        else:
            kept += 1
            n_crops += n
    summary[stem] = {"camera": cam, "identities": kept, "crops": n_crops}
    print(f"DONE {stem} ({cam}): {kept} identities, {n_crops} crops")

with open("/kaggle/working/multicam_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n===== CAMERA-WISE SUMMARY =====")
per_cam = {}
for s in summary.values():
    pc = per_cam.setdefault(s["camera"], [0, 0])
    pc[0] += s["identities"]
    pc[1] += s["crops"]
for cam, (i, c) in sorted(per_cam.items()):
    print(f"{cam}: {i} identities, {c} crops")

shutil.make_archive("/kaggle/working/multicam_crops", "zip", OUT_DIR)
print("\nZip ready: /kaggle/working/multicam_crops.zip")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
4 videos mile:
   /kaggle/input/datasets/payal1238gupta/multi-camera-videos/Cam_1_20260222T000000Z_20260223T000000Z_10fps_chunk_030.mp4
   /kaggle/input/datasets/payal1238gupta/multi-camera-videos/Cam_3_20260222T000000Z_20260223T000000Z_10fps_chunk_037.mp4
   /kaggle/input/datasets/payal1238gupta/multi-camera-videos/Cam_4_20260222T000000Z_20260223T000000Z_10fps_chunk_010.mp4
   /kaggle/input/datasets/payal1238gupta/multi-camera-videos/Cam_8_chunk_001.mp4
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 338ms
 Downloaded lap
Prepared 1 package in 91ms
Installed 1 package in 6ms
 + lap==0.5.13

r

In [3]:
import cv2, glob
from ultralytics import YOLO
vp = [p for p in glob.glob("/kaggle/input/**/*.mp4", recursive=True) if "Cam_4" in p][0]
cap = cv2.VideoCapture(vp)
print("frames:", int(cap.get(7)), "fps:", cap.get(5), "size:", cap.get(3), cap.get(4))
m = YOLO("yolov8n.pt")
for f in [100, 3000, 8000, 15000]:
    cap.set(1, f); ok, img = cap.read()
    if not ok: print(f, "read FAIL"); continue
    b = m(img, classes=[0], conf=0.25, verbose=False)[0].boxes
    hs = (b.xyxy[:, 3] - b.xyxy[:, 1]).int().tolist()
    print(f"frame {f}: {len(b)} persons, box heights {hs}")

frames: 24891 fps: 10.0 size: 1280.0 720.0
frame 100: 0 persons, box heights []
frame 3000: 0 persons, box heights []
frame 8000: 0 persons, box heights []
frame 15000: 0 persons, box heights []


In [4]:
cap = cv2.VideoCapture(vp)
cap.set(1, 5000); ok, img = cap.read(); cv2.imwrite("/kaggle/working/cam4_sample.jpg", img)
found = 0
for f in range(0, 24891, 100):
    cap.set(1, f); ok, img = cap.read()
    if ok and len(m(img, classes=[0], conf=0.25, verbose=False)[0].boxes): found += 1
print("person wale frames:", found, "/ 249")

person wale frames: 0 / 249
